# Notebook 07 — Results Analysis

**Thesis:** RAG-Driven Natural Language Energy Demand Forecasting and Consumption Insight Generator  
**Student:** Zoheb Anwar Hussain (1196931) | **Supervisor:** Ankan Dutta | LJMU MSc UK 2026

---

> **This notebook is deprecated.** All results analysis and benchmarking 
> is now performed in `08_experiments.ipynb` under EXP_10 (Final Comparative Analysis).
> 
> This notebook was used during early pipeline development (Notebooks 01-06) 
> and is retained for traceability only.

### Purpose
This notebook synthesises all evaluation outputs from Notebooks 04–06 into:

1. **Section 1 — Setup & Data Loading** — resolve paths, load all artefacts  
2. **Section 2 — Retrieval Analysis** — grouped bars, radar chart, improvement vs baseline  
3. **Section 3 — Hallucination Analysis** — pass-rate breakdown per pipeline  
4. **Section 4 — RAGAS Analysis** — heatmap with NaN handling, coverage report  
5. **Section 5 — Answer Quality** — word-count distributions  
6. **Section 6 — Unified Scorecard** — composite ranking across all dimensions  
7. **Section 7 — Golden Dataset Audit** — query distribution, known limitations  
8. **Section 8 — Export** — save all figures + generate `docs/07_results_analysis.md`

---
## Section 1 — Setup & Data Loading

In [1]:
# ── stdlib ──────────────────────────────────────────────────────────────────
import sys
import logging
from pathlib import Path

# ── repo root on sys.path ────────────────────────────────────────────────────
REPO_ROOT = Path().resolve().parent          # notebooks/ → repo root
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# ── env / config ─────────────────────────────────────────────────────────────
from dotenv import load_dotenv
load_dotenv(dotenv_path=REPO_ROOT / ".env")

# ── analysis modules ─────────────────────────────────────────────────────────
from src.analysis.metrics_loader import load_all
from src.analysis.aggregators import (
    pipeline_retrieval_summary,
    retrieval_improvement_vs_baseline,
    pipeline_hallucination_summary,
    hallucination_risk_score,
    ragas_summary,
    ragas_coverage,
    answer_length_stats,
    query_source_distribution,
    unified_pipeline_scorecard,
)
from src.analysis.plot_helpers import (
    plot_retrieval_metrics,
    plot_retrieval_radar,
    plot_hallucination_bars,
    plot_ragas_heatmap,
    plot_answer_length_dist,
    plot_scorecard,
    plot_query_distribution,
    save_figure,
)
from src.analysis.report_builder import build_report

# ── plotting ──────────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import pandas as pd

%matplotlib inline
plt.rcParams["figure.dpi"] = 120

logging.basicConfig(level=logging.INFO,
                    format="%(levelname)s | %(name)s | %(message)s")
logger = logging.getLogger("nb07")

print("    Imports OK")
print(f"   REPO_ROOT = {REPO_ROOT}")

    Imports OK
   REPO_ROOT = E:\UpGrad\LJMU_Thesis\Topic_1_Energy_Forecasting\Research_Proposal\Git_Repo\RAG-Based-Energy-Forecasting


In [2]:
# ── Output directories ───────────────────────────────────────────────────────
# Notebooks run from notebooks/ so relative 'outputs/' resolves to
# notebooks/outputs/.  Adjust OUTPUTS_DIR if your layout differs.

OUTPUTS_DIR  = Path("outputs")          # notebooks/outputs/
FIGURES_DIR  = OUTPUTS_DIR / "figures"
DOCS_DIR     = REPO_ROOT / "docs"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Outputs  : {OUTPUTS_DIR.resolve()}")
print(f"Figures  : {FIGURES_DIR.resolve()}")
print(f"Docs     : {DOCS_DIR.resolve()}")

Outputs  : E:\UpGrad\LJMU_Thesis\Topic_1_Energy_Forecasting\Research_Proposal\Git_Repo\RAG-Based-Energy-Forecasting\notebooks\outputs
Figures  : E:\UpGrad\LJMU_Thesis\Topic_1_Energy_Forecasting\Research_Proposal\Git_Repo\RAG-Based-Energy-Forecasting\notebooks\outputs\figures
Docs     : E:\UpGrad\LJMU_Thesis\Topic_1_Energy_Forecasting\Research_Proposal\Git_Repo\RAG-Based-Energy-Forecasting\docs


In [3]:
# ── Load all artefacts ───────────────────────────────────────────────────────
data = load_all(OUTPUTS_DIR)

retrieval_raw  = data["retrieval"]
halluc_raw     = data["hallucination"]
ragas_raw      = data["ragas"]
rag_answers    = data["rag_answers"]
golden_df      = data["golden"]

print("\nLoaded DataFrames:")
for name, df in data.items():
    print(f"  {name:15s}  shape={df.shape}")

WARNING | src.analysis.metrics_loader | retrieval_metrics file not found in outputs\retrieval_results — using hard-coded pilot values.
WARNING | src.analysis.metrics_loader | hallucination file not found in outputs\evaluation — using hard-coded pilot values.
WARNING | src.analysis.metrics_loader | RAGAS file not found in outputs\evaluation — using partial pilot values.
WARNING | src.analysis.metrics_loader | RAG answers file not found in outputs\rag_results.
WARNING | src.analysis.metrics_loader | Golden dataset file not found in outputs\golden_dataset.



Loaded DataFrames:
  retrieval        shape=(3, 6)
  hallucination    shape=(3, 5)
  ragas            shape=(3, 5)
  rag_answers      shape=(0, 5)
  golden           shape=(0, 5)


In [ ]:
# ── Quick sanity checks ──────────────────────────────────────────────────────
print("=== Retrieval (raw) ===")
display(retrieval_raw)

print("\n=== Hallucination (raw) ===")
display(halluc_raw)

print("\n=== RAGAS (raw — NaN expected for partial run) ===")
display(ragas_raw)

---
## Section 2 — Retrieval Analysis

In [ ]:
# ── Pipeline-level summary ───────────────────────────────────────────────────
ret_summary = pipeline_retrieval_summary(retrieval_raw)
print("Pipeline Retrieval Summary (sorted by composite score):")
display(ret_summary)

In [ ]:
# ── % improvement vs dense baseline ─────────────────────────────────────────
ret_improvement = retrieval_improvement_vs_baseline(retrieval_raw, baseline="dense")
print("\n% Improvement over dense baseline:")
display(ret_improvement)

In [ ]:
# ── Plot 1: Grouped bar chart ────────────────────────────────────────────────
fig_ret_bar = plot_retrieval_metrics(retrieval_raw)
plt.show()
save_figure(fig_ret_bar, FIGURES_DIR / "01_retrieval_metrics_bar.png")
print("Saved → figures/01_retrieval_metrics_bar.png")

In [ ]:
# ── Plot 2: Radar chart ──────────────────────────────────────────────────────
fig_ret_radar = plot_retrieval_radar(retrieval_raw)
plt.show()
save_figure(fig_ret_radar, FIGURES_DIR / "02_retrieval_radar.png")
print("Saved → figures/02_retrieval_radar.png")

### 2.1 Interpretation

- **Hierarchical** leads on MRR (0.159) and nDCG (0.075) — the two-stage coarse→fine retrieval helps rank relevant documents higher.
- **Dense** is the balanced baseline, performing consistently across all four metrics.
- **Hybrid** scores lowest at this scale — the BM25 component may be under-tuned on the energy domain vocabulary.
- All absolute values are low (~6–8 %). This is **expected at pilot scale** and directly caused by random context selection in the golden dataset (the ground-truth answers reference context the retrievers were not primed to fetch). BM25-guided context selection (planned) will substantially raise these figures.

---
## Section 3 — Hallucination Analysis

In [ ]:
# ── Aggregated pass rates ────────────────────────────────────────────────────
hal_summary = pipeline_hallucination_summary(halluc_raw)
print("Hallucination pass-rate summary:")
display(hal_summary)

print("\nRisk scores (lower = better):")
display(hallucination_risk_score(halluc_raw))

In [ ]:
# ── Plot 3: Hallucination bars ───────────────────────────────────────────────
fig_hal = plot_hallucination_bars(hal_summary)
plt.show()
save_figure(fig_hal, FIGURES_DIR / "03_hallucination_bars.png")
print("Saved → figures/03_hallucination_bars.png")

### 3.1 Interpretation

| Check | What it tests |
|---|---|
| **Include-pass** | Does the answer contain key facts from the retrieved context? |
| **Exclude-pass** | Does the answer avoid introducing facts *not* in the context? |
| **Overall-pass** | Both checks pass simultaneously |

- **Exclude-pass ≥ 95 %** across all pipelines: the LLM rarely hallucinates outright fabrications — strong grounding behaviour.
- **Include-pass ≈ 56–65 %**: the model does not always surface every key fact from the context into the answer, which is typical for abstractive generation.
- **Hierarchical** achieves the best overall-pass (60 %), consistent with its stronger retrieval — better context → fewer omissions.

---
## Section 4 — RAGAS Analysis

In [ ]:
# ── Coverage report ──────────────────────────────────────────────────────────
coverage = ragas_coverage(ragas_raw)
print("RAGAS metric coverage (partial run):")
display(coverage)

In [ ]:
# ── Aggregated scores ────────────────────────────────────────────────────────
ragas_agg = ragas_summary(ragas_raw)
print("\nRAGAS scores per pipeline (NaN = rate-limited):")
display(ragas_agg)

In [ ]:
# ── Plot 4: RAGAS heatmap ─────────────────────────────────────────────────────
fig_ragas = plot_ragas_heatmap(ragas_agg)
plt.show()
save_figure(fig_ragas, FIGURES_DIR / "04_ragas_heatmap.png")
print("Saved → figures/04_ragas_heatmap.png")

### 4.1 Interpretation

- Only `dense` × `answer_relevancy` succeeded (0.822) before the Groq 100 K token/day limit was exhausted.
- **0.822 answer_relevancy** is a strong result: even with imperfect retrieval, Llama 3.3 70B produces answers that closely address the question.
- Re-run Notebook 06 after the 24-hour token reset to complete the evaluation.

> **TODO (high priority):** Re-run with fresh token quota and update this notebook.

---
## Section 5 — Answer Quality

In [ ]:
# ── Length statistics ────────────────────────────────────────────────────────
len_stats = answer_length_stats(rag_answers)
if not len_stats.empty:
    print("Answer word-count statistics per pipeline:")
    display(len_stats)
else:
    print("RAG answers not found in outputs — skipping length analysis.")
    print("   (This is normal if notebook 05 saved answers to a different path.)")

In [ ]:
# ── Plot 5: Answer length distribution ──────────────────────────────────────
fig_len = plot_answer_length_dist(rag_answers)
plt.show()
save_figure(fig_len, FIGURES_DIR / "05_answer_length_dist.png")
print("Saved → figures/05_answer_length_dist.png")

### 5.1 Interpretation

- Mean answer length: **94 words** across all 93 generation calls — concise but substantive for energy-domain queries.
- Zero generation errors demonstrates model stability across diverse query types (daily, weekly, seasonal, appliance-level).
- Variance across pipelines (if any) reflects prompt differences rather than model capability.

---
## Section 6 — Unified Scorecard

In [ ]:
# ── Build scorecard ──────────────────────────────────────────────────────────
scorecard = unified_pipeline_scorecard(
    retrieval_df=retrieval_raw,
    halluc_df=hal_summary,
    ragas_df=ragas_agg,
)

print("Unified Pipeline Scorecard:")
display(scorecard)

In [ ]:
# ── Plot 6: Scorecard bar ─────────────────────────────────────────────────────
fig_sc = plot_scorecard(scorecard)
plt.show()
save_figure(fig_sc, FIGURES_DIR / "06_scorecard.png")
print("Saved → figures/06_scorecard.png")

### 6.1 Interpretation

The scorecard normalises and averages:
- **Retrieval composite** (mean of recall, precision, MRR, nDCG) — higher is better
- **(1 − hallucination risk)** — higher means less hallucination

Ranking interpretation will be updated once RAGAS scores are complete.

---
## Section 7 — Golden Dataset Audit

In [ ]:
# ── Query distribution ───────────────────────────────────────────────────────
if not golden_df.empty:
    dist = query_source_distribution(golden_df)
    print("Query distribution by source × granularity:")
    display(dist)
    print(f"\nTotal queries: {len(golden_df)}")
else:
    print("Golden dataset not found — skipping distribution analysis.")

In [ ]:
# ── Plot 7: Query distribution ───────────────────────────────────────────────
fig_qd = plot_query_distribution(golden_df)
plt.show()
save_figure(fig_qd, FIGURES_DIR / "07_query_distribution.png")
print("Saved → figures/07_query_distribution.png")

### 7.1 Known Limitation: Random Context Selection

The 50 golden-dataset queries were generated using **random context sampling** from the KB (pragmatic pilot approach).  This means:

1. The ground-truth answers are anchored to *randomly chosen* KB documents.
2. The retrieval pipelines — which do semantic search — will often retrieve *different but equally valid* documents.
3. This mismatch causes the retrieval metrics to appear artificially low (6–8 %).

**Planned fix:** Replace `src/golden_dataset/context_selector.py` with BM25-ranked selection so the ground-truth context matches what a retriever would naturally surface.  Confirm approach with supervisor before re-running.

---
## Section 8 — Export

In [ ]:
# ── Save all summary tables as CSV ───────────────────────────────────────────
TABLES_DIR = OUTPUTS_DIR / "analysis_tables"
TABLES_DIR.mkdir(parents=True, exist_ok=True)

ret_summary.to_csv(TABLES_DIR / "retrieval_summary.csv", index=False)
hal_summary.to_csv(TABLES_DIR / "hallucination_summary.csv", index=False)
ragas_agg.to_csv(TABLES_DIR  / "ragas_summary.csv",         index=False)
scorecard.to_csv(TABLES_DIR  / "scorecard.csv",              index=False)

print("Summary tables saved to", TABLES_DIR.resolve())

In [ ]:
# ── Generate Markdown report ─────────────────────────────────────────────────
report_md = build_report(
    retrieval_df=ret_summary,
    halluc_df=hal_summary,
    ragas_df=ragas_agg,
    scorecard_df=scorecard,
    rag_df=rag_answers,
)

report_path = DOCS_DIR / "07_results_analysis.md"
report_path.write_text(report_md, encoding="utf-8")

print(f"Markdown report saved → {report_path.resolve()}")
print(f"   ({len(report_md):,} characters)")

In [ ]:
# ── Figures inventory ────────────────────────────────────────────────────────
figs = sorted(FIGURES_DIR.glob("*.png"))
print(f"\nFigures saved ({len(figs)} total):")
for f in figs:
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:45s}  {size_kb:6.1f} KB")

---
## Summary

| Dimension | Best pipeline | Key metric |
|---|---|---|
| Retrieval (MRR) | **Hierarchical** | 0.159 |
| Retrieval (nDCG) | **Hierarchical** | 0.075 |
| Hallucination resistance | **Hierarchical** | 60 % overall-pass |
| Answer relevancy (RAGAS) | **Dense** | 0.822 (only available score) |

### Immediate next steps

1. 🔴 **Re-run RAGAS** after Groq token reset and update Section 4 + scorecard
2. 🔴 **Discuss BM25 context selection** with supervisor → update `context_selector.py`
3. 🟡 **Scale KB** to 200+ docs/type and re-run full pipeline
4. 🟡 **Write unit tests** for `src/analysis/` and `src/knowledge_base/`
5. 🟢 **Re-enable ChromaDB** on Linux/Colab for vector store comparison